# MethaneScope AI · Lesson 2.10
## Multi-granule NASA EMIT V002 acquisition

**Prerequisites:** Lessons 2.3–2.9 complete; NASA Earthdata account; `25 passed` test suite; one original EMIT granule already stored in PostGIS.

**Learning goals:** Discover multiple real plume granules; inspect metadata URLs; protect existing files; download a *small* batch after explicit approval; rerun Lesson 2.3 QA; ingest only ACCEPTED records; inspect spatial/temporal distributions.

**Critical scientific limitation:** `EMITL2BCH4PLM.002` is a **plume detections** collection, not a complete record of all valid observations or non-detections. You cannot calculate defensible source persistency from plume-only records. That requires an observation-opportunities dataset and detection sensitivity/quality analysis.

We will not assume the newly discovered granules are from the same source, nor that multiple plume detections imply independent emission events.

NASA official sources:
- https://doi.org/10.5067/EMIT/EMITL2BCH4PLM.002
- https://nasa.github.io/LPDAAC-Data-Resources/external/How_to_find_and_access_EMIT_data.html
- https://earthaccess.readthedocs.io/en/stable/user/quick-start/

### 1. Environment and safety

Run this notebook from the **MethaneScope project root**. We are installing only an additional module `methanescope/acquire.py`, not replacing your previous pipeline.

From Git Bash (not a notebook cell):

```bash
source .venv/Scripts/activate
python -m pip install earthaccess
python -m pytest -q tests/test_acquire.py
```

The source code is safe by default: `plan` only queries NASA catalogue metadata; `download` without `--execute` performs **no downloads**; `download --execute` is required for actual network downloads.

In [ ]:
import sys
from pathlib import Path
from IPython.display import display
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'methanescope' / 'acquire.py').is_file():
    raise RuntimeError(f'Open JupyterLab from MethaneScope project root, not: {ROOT}')
print('Project:', ROOT)
print('Python:', sys.executable)
print('Existing NASA rasters:', len(list((ROOT / 'data' / 'raw' / 'nasa_emit').rglob('*.tif'))))

### 2. Understand acquisition versus ingestion

- **Discovery:** Read NASA Common Metadata Repository (CMR) records; do not download.
- **Acquisition:** Download **matching** COG raster and `CH4PLMMETA` JSON into a staging folder and validate file structure. Never overwrite prior data.
- **Quality control:** Rerun your existing Lesson 2.3 notebook. Only the `ACCEPTED` manifest is eligible for database ingestion.
- **Database ingestion:** Use the **existing** Lesson 2.5 pipeline or Lesson 2.9 safe scheduler; avoid changing database schema.

### 3. Discover first, without downloading anything

This makes a metadata search of up to 40 NASA granules, and selects at most three *new* granules. These are **candidate downloads**, not verified scientific detections.

Search may take several seconds and requires internet access. The time interval is intentionally omitted for this first discovery so NASA can return available Version 2 plume granules.

In [ ]:
import subprocess

cmd = [
    sys.executable, '-m', 'methanescope.acquire',
    'plan', '--search-limit', '40', '--max-new', '3'
]
result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
print(result.stdout or result.stderr)
if result.returncode:
    raise RuntimeError(f'NASA CMR discovery failed with exit code {result.returncode}')

### 4. Inspect proposed granules and file pairs

The plan JSON includes all catalogue results, but does not display credentials or download tokens. `AVAILABLE` means a complete pair *appears to exist* in NASA's catalogue; further local file QA is necessary.

In [ ]:
import json
plan_path = ROOT / 'outputs' / 'acquisition' / 'last_plan.json'
plan = json.loads(plan_path.read_text(encoding='utf-8'))

print('Catalogue matches:', plan['searched'])
print('Proposed new granules:', len(plan['planned_new_ids']))
print('Selected IDs:')
for gid in plan['planned_new_ids']:
    print(' -', gid)

info = pd.DataFrame(plan['items'])
if not info.empty:
    display(info[['id','status']].head(40))
else:
    print('No granules returned; try another temporal window or inspect search errors')

### 5. Download a small, explicitly approved batch

**Nothing will download until you deliberately change `ALLOW_DOWNLOAD` to `True`.**

NASA Earthdata Login may prompt you to sign in on your computer. Never paste credentials or tokens into shared notebooks or chat. Start with a maximum of 3 granules to avoid excessive downloads, and check available free disk space first.

On a successful run, the downloader saves the original filenames under `data/raw/nasa_emit/`; failed or ambiguous files remain in `data/staging/emit_lesson210/` for investigation.

In [ ]:
ALLOW_DOWNLOAD = False   # Deliberately change to True after reviewing the plan and disk space

if not ALLOW_DOWNLOAD:
    print('SAFE STOP: No files downloaded. Set ALLOW_DOWNLOAD = True when ready.')
else:
    cmd = [sys.executable, '-m', 'methanescope.acquire', 'download',
           '--search-limit', '40', '--max-new', '3', '--execute']
    result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    print(result.stdout or result.stderr)
    if result.returncode:
        raise RuntimeError(f'Acquisition returned exit code {result.returncode}; inspect outputs/acquisition/last_download.json')

### 6. Inspect actual download results

Remember: a **downloaded** granule is *not* an **accepted** ingestion record. We must rerun the full Lesson 2.3 quality-control process.

In [ ]:
report_path = ROOT / 'outputs' / 'acquisition' / 'last_download.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print('Downloaded statuses:', [x['status'] for x in report.get('downloaded', [])])
    display(pd.DataFrame(report.get('downloaded', [])))
else:
    print('No download report yet (normal until the download cell has been explicitly enabled).')

### 7. Quality-control gate: rerun Lesson 2.3 **after the download**

1. Back up the current accepted manifest in Git Bash:

```bash
cp outputs/emit_batch/emit_accepted_for_ingestion.csv outputs/emit_batch/emit_accepted_before_lesson210.csv
```

2. Open **`MethaneScope_Lesson_2_3_NASA_Metadata_Pairing_Fixed.ipynb`**.
3. Confirm `MODE = 'real'`.
4. Select **Kernel → Restart Kernel and Run All Cells**.
5. Inspect `ACCEPTED / REVIEW / REJECTED / DUPLICATE / CONFLICT` and `outputs/emit_batch/emit_batch_issues.json`.
6. **Do not manually change `REVIEW` to `ACCEPTED`.** Fix the true issue first.

If the new NASA granules are compatible with the existing validator, accepted rows should increase. The exact count cannot be assumed until the files have been downloaded and checked.

In [ ]:
manifest = ROOT / 'outputs' / 'emit_batch' / 'emit_accepted_for_ingestion.csv'
if manifest.exists():
    accepted = pd.read_csv(manifest, keep_default_na=False)
    print('Accepted records in most recent manifest:', len(accepted))
    if len(accepted):
        display(accepted[['granule_id', 'status', 'geojson_path']])
else:
    print('Accepted manifest missing; run Lesson 2.3 QC first.')

### 8. Revalidate and ingest using the existing tested application

After Lesson 2.3 reports the expected accepted records, run in **Git Bash**:

```bash
docker compose -f docker-compose.postgis.yml up -d
python -m methanescope.pipeline discover
python -m methanescope.pipeline validate
python -m methanescope.scheduler plan
python -m methanescope.scheduler run --execute
python -m methanescope.pipeline status
```

Existing record(s) should be **SKIPPED**; new accepted granules should be **INSERTED**. If the output contains an ERROR, REVIEW or CONFLICT, stop and examine the audit reports before proceeding.

### 9. Query actual spatial and temporal records from PostGIS

**Only run this after the new ingestion step finishes.** The queries are read-only.

In [ ]:
from methanescope.store import connection

with connection(ROOT) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT g.granule_id, g.acquisition_time,
                   ROUND(g.max_enhancement_ppm_m::numeric, 2),
                   ST_AsText(ST_Centroid(g.footprint)),
                   COUNT(f.id) AS feature_count
            FROM satellite_granules g
            LEFT JOIN plume_features f ON f.granule_pk=g.id
            GROUP BY g.id, g.granule_id, g.acquisition_time,
                     g.max_enhancement_ppm_m, g.footprint
            ORDER BY g.acquisition_time
        """)
        rows = cur.fetchall()
        cols = [d.name for d in cur.description]

summary = pd.DataFrame(rows, columns=cols)
print('Granules stored:', len(summary))
display(summary)

### 10. Scientific interpretation checkpoint

Answer these questions in your study notes:

1. Why is `CH4PLMMETA` metadata paired with the `CH4PLM` raster by granule ID rather than by identical filenames?
2. Why do we have distinct `DOWNLOADED`, `ACCEPTED`, `INSERTED`, and `SKIPPED` statuses?
3. Why is a methane enhancement in **ppm m** not an emission rate in **kg/h**?
4. Why can't we estimate an emitter's **persistency** from a plume-only catalogue?
5. Why can two plume files near the same geographical location represent one event, multiple events, or separate sources?

**Milestone:** A reviewed collection of multiple genuine NASA observations stored in PostGIS with preserved provenance and without duplicates.

**Next:** Spatial-temporal exploration, potential source association, and collecting observation opportunities including non-detections before persistency analysis.